In [ ]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(".").resolve()
DA_MAPPINGS_DIR = BASE_DIR / "da_mappings" / "da_mappings_obx"


In [ ]:
from IPython.display import display, HTML

rows = []

for fp in sorted(DA_MAPPINGS_DIR.glob("*.json")):
    data = json.loads(fp.read_text(encoding="utf-8"))
    firm_id = fp.stem

    for var_block in data.get("variables", []):
        variable = var_block.get("variable")
        final_choice = var_block.get("final_choice", [])

        if final_choice:  # only keep non-empty final_choice
            for ch in final_choice:
                rows.append({
                    "firm_id": firm_id,
                    "variable": variable,
                    "sheet_name": ch.get("sheet_name", ""),
                    "row_label": ch.get("row_label", ""),
                    "needs_manual_review": var_block.get("needs_manual_review", False),
                    "notes": var_block.get("notes", ""),
                })

df_nonempty = pd.DataFrame(rows)

if df_nonempty.empty:
    print("No non-empty final_choice found.")
else:
    df_show = df_nonempty.sort_values([ "firm_id","variable"]).reset_index(drop=True)

    html_table = df_show.to_html(index=False, escape=False)

    display(HTML(f"""
    <div style="
        max-height: 600px;
        overflow-y: auto;
        overflow-x: auto;
        border: 1px solid #ccc;
        padding: 6px;
        background: black;
    ">
        {html_table}
    </div>
    """))

firm_id,variable,sheet_name,row_label,needs_manual_review,notes
ACG1V.HE,XSGA_DA,Income Statement,Depreciation and amortization,False,"OK. The selected XSGA_COMPONENTS bucket is 'Selling, General & Administrative Expenses'. The separate 'Depreciation and amortization' line appears outside and alongside that selected SG&A row, so it is not already embedded in the selected parent bucket. Per the summary-vs-components rule, only the total D&A row is chosen, not its components."
BIOBV.HE,XSGA_DA,Income Statement,Depreciation and Amortization,True,"ADDED MANUALLY. No clear populated SG&A-only D&A row is visible above EBIT. The generic row 'Depreciation and Amortization' may be separate operating D&A, but it is not clearly attributable only to XSGA and may include COGS. To avoid double counting and misallocation, no XSGA_DA row is selected, but manual review is warranted."
BOREO.HE,XSGA_DA,Income Statement,Depreciation and Amortization,False,OK. Use the summary operating D&A row only. It appears separate from the selected XSGA component rows and should be added later to avoid omitting overhead D&A. Do not also include component rows.
CONSTI.HE,XSGA_DA,Income Statement,Depreciation,False,"OK. A separate operating D&A summary row is visible above EBIT and appears outside the selected XSGA component rows, so it likely should be added separately rather than treated as embedded. However, the row is generic and not explicitly split between SG&A and production/COGS, so manual review is warranted before assigning the full amount to XSGA_DA."
CTH1V.HE,XSGA_DA,Income Statement,"Depreciation, amortization and write downs",True,"ADDED MANUALLY. A separate operating D&A summary row exists and appears outside the selected XSGA bucket ('Operating expenses' / 'Selling, General & Admin'), which suggests D&A may need separate treatment. But the row is not labelled as SG&A-specific and may include non-SG&A operating D&A (potentially production-related). To avoid misclassification and double counting, leave blank pending manual review."
CTY1S.HE,XSGA_DA,Income Statement,Depreciation and amortisation,False,"ADDDED MANUALLY. The selected XSGA bucket is 'Administrative Expenses'. The populated row 'Depreciation and amortisation' is shown as part of the administrative-expense breakdown, so it is best treated as already included in the selected parent row. Therefore no separate XSGA_DA add-back row should be selected."
DIGIA.HE,XSGA_DA,Income Statement,Depreciation and amortisation,False,"MANUALLY ADDED.No SG&A-specific or R&D-specific D&A row is disclosed above EBIT. The explicit D&A lines appear to be total operating D&A and components, so they should not be added separately to the selected XSGA bucket."
DIGIGR.HE,XSGA_DA,Income Statement,Depreciation and impairment,False,"ADDED MANUALLY. Although explicit D&A rows appear separately at the operating level, they are not attributed to SG&A versus COGS/R&D. Because the selected XSGA bucket is built from specific component rows rather than a parent total, adding unlabeled company-wide depreciation/amortisation would risk distortion. Conservative choice is to add nothing separately."
EEZY.HE,XSGA_DA,Income Statement,Amortization of Intangibles excluding Goodwill,False,"OK. The selected XSGA bucket consists of Personnel Expenses plus detailed Other operating expenses components. The statement presents amortization and depreciation as separate operating lines alongside those buckets, suggesting they are not already embedded in the selected XSGA rows and should be added separately. Used the summary component rows 'Amortization of Intangibles excluding Goodwill' and 'Depreciation' rather than their more granular sub-lines to avoid double counting. Manual review is warranted because the excerpt does not explicitly label these D&A rows as SG&A-only; they may contain broader operating D&A."
EEZY.HE,XSGA_DA,Income Statement,Depreciation,False,"OK. The selected XSGA bucket consists of Personnel Expenses plus detailed O

In [3]:
df_manual = df_nonempty[df_nonempty["needs_manual_review"] == True]

print(f"Number of rows:                    {len(df_nonempty)}")
print(f"Unique tickers (firm_id):          {df_nonempty['firm_id'].nunique()}")
print(f"\nNeeds manual review (total rows):  {len(df_manual)}")
print(f"Needs manual review (unique tickers): {df_manual['firm_id'].nunique()}")
print(f"\nAll tickers: {sorted(df_nonempty['firm_id'].unique().tolist())}")

Number of rows:                    77
Unique tickers (firm_id):          53

Needs manual review (total rows):  16
Needs manual review (unique tickers): 9

All tickers: ['ACG1V.HE', 'BIOBV.HE', 'BOREO.HE', 'CONSTI.HE', 'CTH1V.HE', 'CTY1S.HE', 'DIGIA.HE', 'DIGIGR.HE', 'EEZY.HE', 'ELISA.HE', 'ENENTO.HE', 'ESENSE.HE', 'ETTE.HE', 'HKFOODS.HE', 'HONBS.HE', 'ICP1V.HE', 'INVEST.HE', 'KELAS.HE', 'KESKOB.HE', 'LAMOR.HE', 'LEHTO.HE', 'LUMO.HE', 'MARAS.HE', 'METSB.HE', 'OLVAS.HE', 'OPTOMED.HE', 'ORIOLA.HE', 'PAMPALO.HE', 'QPR1V.HE', 'QTCOM.HE', 'RAP1V.HE', 'RAUTE.HE', 'REBL.HE', 'REG1V.HE', 'RELAIS.HE', 'REMEDY.HE', 'ROBIT.HE', 'SAGCV.HE', 'SANOMA.HE', 'SIILI.HE', 'SITOWS.HE', 'SOLTEQ.HE', 'SRV1V.HE', 'SSH1V.HE', 'TEM1V.HE', 'TNOM.HE', 'TOIVO.HE', 'TOKMAN.HE', 'TRH1V.HE', 'TTALO.HE', 'TULAV.HE', 'VALMT.HE', 'WETTERI.HE']
